# 第8周｜三只股票的综合比较

**学习目标：**串联分组、变形、相关性、绘图与滚动计算。

先预测、再运行、最后解释。允许查课件，重点是读懂和修改代码。

**使用方式：**选择课程 `.venv` 内核，从上到下运行。示例可以直接运行；练习单元格中的注释是待补全位置，取消注释并填写。每题按本周准备单元格独立取数，不依赖上一题答案。参考答案默认折叠，先独立完成再展开。

## 准备与小示例

使用第11章CSMAR本地数据的2025年三股摘录：000001、600000、600519。每行是一只股票一个交易日的观测。

| 字段 | 含义 |
|---|---|
| Stkcd / Trddt | 六位股票代码 / 交易日期 |
| Clsprc | 未复权收盘价，元 |
| Dretwd | 含现金红利再投资的日收益率，小数，0.01=1% |
| Dnshrtrd / Dnvaltrd | 成交股数（股）/ 成交金额（元） |

教师已按第11章规则筛选A股、非填充、有成交、正价格和有效收益记录。完整来源与生成方法见[数据说明](../data/README.md)。若CSV尚未准备，请教师运行提取脚本；不要用随机数据替换。

以下路径、字体和读取代码由教师提供，无需默写。每周独立读取同一份样本，不依赖上一周生成的文件。

本周以相同日期区间比较三只股票。这里比较单只股票，不构建投资组合或交易策略。

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 教师提供：从Notebook目录、上机练习目录或课程根目录运行均可。
cwd = Path.cwd()
if (cwd / "上机练习").is_dir():
    LAB = cwd / "上机练习"
elif cwd.name == "notebooks":
    LAB = cwd.parent
else:
    LAB = cwd
DATA = LAB / "data"
OUTPUTS = LAB / "outputs"
OUTPUTS.mkdir(exist_ok=True)

df = pd.read_csv(DATA / "stocks_2025.csv", dtype={"Stkcd": str}, parse_dates=["Trddt"])
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
TARGET = "000001"
stock = df.loc[df["Stkcd"] == TARGET].sort_values("Trddt").set_index("Trddt")

In [2]:
toy = pd.Series([1, 2, 3, 4])
print(toy.rolling(3).mean())
display(df.groupby("Stkcd")["Trddt"].agg(["min", "max", "count"]))

0    NaN
1    NaN
2    2.0
3    3.0
dtype: float64


,min,max,count
Stkcd,,,
000001,2025-01-02,2025-12-31,243
600000,2025-01-02,2025-12-31,243
600519,2025-01-02,2025-12-31,243


**输出解释：**长度3的滚动窗口前两项不足3个观测，因此为NaN。真实数据比较前先看日期范围与有效数量，还要检查中间日期是否对应。

## 练习1｜按股票生成统计表

按股票统计日收益的count、mean、std，并按std从低到高排序。mean较大能否说明累计收益一定更大？

In [3]:
# summary = df.groupby(...)[...].agg([...])
# display(summary.sort_values(...))

**我的解释：**（填写预测、实际结果或原因。）

<details><summary>参考答案（先自己动手）</summary>

```python
summary = df.groupby("Stkcd")["Dretwd"].agg(["count", "mean", "std"])
display(summary.sort_values("std"))
```

每行现在代表一只股票；平均日收益不包含完整的复合路径信息，累计收益需要连乘计算。

</details>

## 练习2｜从长表到宽表

用pivot生成日期为行、股票为列、日收益为值的宽表，按日期排序。解释转换前后的形状。

In [4]:
# wide = df.pivot(index=..., columns=..., values=...).sort_index()
# display(wide.head())
# print(df.shape, wide.shape)

**我的解释：**（填写预测、实际结果或原因。）

<details><summary>参考答案（先自己动手）</summary>

```python
wide = df.pivot(index="Trddt", columns="Stkcd", values="Dretwd").sort_index()
display(wide.head())
print(df.shape, wide.shape)
```

长表每行是一只股票一天，宽表每行是同一天、三列分别是三股收益。pivot要求股票与日期组合唯一。

</details>

## 练习3｜共同日期与相关性

检查宽表缺失值，取三股都有收益的日期计算相关系数。说明对角线的1与非对角线元素。

In [5]:
wide = df.pivot(index="Trddt", columns="Stkcd", values="Dretwd").sort_index()
# print(wide.isna().sum())
# common = wide.dropna()
# display(common.corr())

**我的解释：**（填写预测、实际结果或原因。）

<details><summary>参考答案（先自己动手）</summary>

```python
wide = df.pivot(index="Trddt", columns="Stkcd", values="Dretwd").sort_index()
print(wide.isna().sum())
common = wide.dropna()
print("全部日期与共同日期：", len(wide), len(common))
display(common.corr())
```

对角线为自身相关系数1；非对角线描述共同日期日收益的线性关联，不是因果关系。dropna用于统一相关性样本，不能随意删日后把累计收益称为完整持有期收益。

</details>

## 练习4｜比较累计收益曲线

确认三股日期完整一致后，补全累计收益计算并画在同一张图上。打印期末累计收益并排序。

In [6]:
wide = df.pivot(index="Trddt", columns="Stkcd", values="Dretwd").sort_index()
print("是否全部完整：", wide.notna().all().all())
# cumulative = ...
# ax = (cumulative * 100).plot()
# 请补全标题、单位和期末排序

是否全部完整： True


**我的解释：**（填写预测、实际结果或原因。）

<details><summary>参考答案（先自己动手）</summary>

```python
wide = df.pivot(index="Trddt", columns="Stkcd", values="Dretwd").sort_index()
print("是否全部完整：", wide.notna().all().all())
assert wide.notna().all().all(), "先检查日期覆盖，不要直接删除中间缺失日计算完整持有期收益。"
cumulative = (1 + wide).cumprod() - 1
ax = (cumulative * 100).plot()
ax.set(title="三股2025年含红利累计收益｜含首日收益", xlabel="交易日期", ylabel="累计收益（%）")
ax.figure.tight_layout()
plt.show()
print(cumulative.iloc[-1].sort_values(ascending=False))
```

基准均为首个样本日收益发生前的本金1，包含首日日收益；不能用三只股票原始价格水平直接比较回报。此处assert只用于确认比较成立的日期条件。

</details>

## 练习5｜读懂滚动波动

对000001计算20期滚动日收益标准差并绘图。把窗口改为5，解释开头缺失数量和曲线变化。不要求年化。

In [7]:
# r = stock["Dretwd"]
# rolling_std = r.rolling(...).std()
# print(rolling_std.head(22))
# 请绘图

**我的解释：**（填写预测、实际结果或原因。）

<details><summary>参考答案（先自己动手）</summary>

```python
r = stock["Dretwd"]
rolling_std = r.rolling(20).std()
print(rolling_std.head(22))
fig, ax = plt.subplots()
ax.plot(rolling_std.index, rolling_std * 100)
ax.set(title=f"{TARGET}｜2025年20期滚动日收益标准差", xlabel="交易日期", ylabel="日收益标准差（%）")
fig.tight_layout()
plt.show()
```

20期窗口前19个值为空，5期窗口前4个为空。窗口按有效观测计数，不是自然日；较短窗口通常变化更快。此图没有乘sqrt(252)，不是年化波动率。

</details>

## 练习6｜写出有边界的结论

用约150字说明：样本区间、股票范围、累计收益与波动排序、相关性的一项观察，以及为何不能直接推断未来。可查阅前题结果，不需要新模型。

In [8]:
# 本题无需新增代码，请在上方“我的解释”中作答。

**我的解释：**（填写预测、实际结果或原因。）

<details><summary>参考答案（先自己动手）</summary>

```python
# 用前面运行得到的结果写文字，不编造尚未计算的数值。
```

参考结构：“本次使用2025年三只股票的含红利日收益……；累计收益排序为……，日收益标准差排序为……；共同日期上……两股相关性较高。结果描述这一样本，未扣交易成本，不能保证未来收益，也不能代表整个A股市场。”应按实际输出填写，不只抄模板。

</details>

## 提交与自查

- 提交分组统计表、相关系数表、累计收益比较图与滚动标准差图。
- 完成约150字分析，写明样本和解释边界。

提交本周 Notebook，保留你的代码、输出和文字解释。重启内核并按顺序运行一次；不要只提交截图。如使用AI辅助，请注明用在哪一题，并确保能解释关键行、现场修改一个参数。